### Подготовка датасета по показателю надой молока

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from statsmodels.graphics.tsaplots import plot_acf
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

from pylab import rcParams
from IPython.display import display
import math
from prophet import Prophet
pd.set_option('display.max_columns', 130)


import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter("ignore", category=InterpolationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)



In [2]:
df = pd.read_excel("../../Data cleansing/output data/Просуммированные по категориям с доп регрессорами.xlsx")
df.head(5)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
0,Верблюды,АКМОЛИНСКАЯ ОБЛАСТЬ,0.00,0.00,0.40,0.00,0.00,0.00,0.09,1.00,0.00,0.00,0.20,0.00,0.00,0.18,0.28,0.00,0.00,0.40,0.00,0.00,0.65,0.00,0.31,1.00,0.00,0.00,0.00,0.00,0.00,2.01,0.00,1.20,0.00,0.00,2.18,0.39,0.00,0.00,1.04,0.00,0.14,2.08,0.00,0.00,0.00,0.00,0.00,0.30,0.00,0.00,0.00,0.66,0.00,0.33,0.00,0.9,0.00,0.00,0.00,10.08,0.00,0.0,0.00,0.0,0.00,0.54,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.40,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00
1,Верблюды,АКТЮБИНСКАЯ ОБЛАСТЬ,101.98,67.47,374.84,115.59,218.72,14.15,19.77,3.00,39.16,46.16,238.56,463.35,109.04,72.94,384.96,114.35,221.89,10.93,18.79,3.50,38.53,46.64,216.08,472.07,110.32,68.17,380.15,127.12,217.61,14.98,21.26,7.14,44.63,51.78,239.43,513.33,115.70,69.40,371.53,130.17,218.41,14.59,21.76,7.19,45.97,55.48,252.68,527.66,117.94,70.37,385.75,117.80,218.45,15.54,21.78,7.2,47.06,58.02,257.54,543.35,119.80,71.3,396.90,121.8,228.10,15.90,21.9,7.2,63.00,59.90,260.20,552.10,121.60,71.8,398.40,128.50,228.80,13.10,22.50,7.20,63.90,61.90,150.70,554.90,121.30,73.3,404.20,128.70,234.90,13.60,23.60,6.3,49.80,62.4,153.70,552.8,118.07,69.17,384.21,121.01,223.46,11.76,21.39,5.52,48.18,61.39,150.87,531.79,119.50,72.60,391.0,126.1,229.10,12.2,24.10,5.7,49.87,63.00,154.3,538.40
2,Верблюды,АЛМАТИНСКАЯ ОБЛАСТЬ,1.00,0.20,51.60,25.40,0.00,61.90,76.87,95.57,16.09,0.10,10.80,46.78,15.06,13.70,131.80,9.20,17.89,130.12,2.00,0.00,4.99,2.06,4.90,44.48,15.06,23.94,41.60,19.54,3.00,15.14,18.50,14.82,14.52,12.60,9.92,43.11,20.72,2.96,15.24,0.00,1.00,20.03,3.44,7.77,117.25,1.00,0.00,20.34,21.95,3.04,17.91,26.81,9.16,18.80,13.37,0.0,17.29,15.90,4.69,23.10,10.85,9.7,42.75,0.0,6.21,21.10,4.0,2.6,28.10,4.53,79.50,93.18,23.70,11.2,16.00,2.00,3.10,19.62,2.30,4.85,11.18,2.80,9.65,36.44,10.60,11.5,26.10,7.20,12.75,11.20,11.20,27.0,25.26,17.9,21.90,12.5,16.55,16.81,22.78,12.10,12.44,6.85,39.65,17.22,8.85,27.48,0.48,29.72,18.90,17.40,12.5,16.4,11.60,16.7,15.70,24.0,7.70,6.90,5.4,12.60
3,Верблюды,АТЫРАУСКАЯ ОБЛАСТЬ,213.89,167.70,306.60,164.97,342.57,192.00,43.60,113.03,262.37,193.97,308.17,1087.33,325.10,190.60,301.10,154.84,328.50,220.30,66.10,118.20,234.90,196.51,270.92,974.18,303.30,167.41,323.54,182.34,349.10,249.40,57.57,88.40,221.21,199.63,278.80,964.02,305.22,159.62,326.87,159.50,367.20,257.50,60.01,78.10,253.27,332.00,355.05,989.86,280.08,165.05,334.51,134.85,367.97,321.22,126.22,93.7,263.10,366.66,348.54,1041.38,293.40,154.8,339.44,143.6,405.80,308.35,88.5,130.5,575.30,458.70,357.62,998.10,292.76,188.3,487.02,129.28,421.44,351.20,95.77,97.54,624.80,631.65,499.30,1179.66,286.67,189.8,513.16,171.46,402.10,432.22,121.53,126.2,637.50,651.7,482.94,1412.2,307.85,200.59,466.60,182.63,414.56,405.19,184.80,154.96,717.92,719.01,522.83,1164.77,323.81,423.76,441.9,219.1,487.69,446.9,191.43,133.5,730.34,709.68,537.6,930.28
4,Верблюды,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,0.30,0.00,0.00,1.14,6.12,4.70

In [3]:
df['Показатель'].unique()

array(['Верблюды', 'КРС', 'Лошади', 'Молоко', 'Овцы и козы', 'Птица',
       'Свиньи', 'Яйца', 'Температура', 'Поголовье: КРС',
       'Поголовье: лошади', 'Поголовье: овцы и козы', 'Поголовье: свиньи',
       'Поголовье: птица домашняя', 'Поголовье: верблюды', 'Осадки',
       'Цена: Говядина', 'Цена: Баранина', 'Цена: Молоко', 'Цена: Яйца'],
      dtype=object)

In [4]:
df_milk = df[df['Показатель'].isin(['Молоко', 'Температура', 'Осадки', 'Поголовье: КРС', 'Цена: Молоко'])]
df_milk.sample(10)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
384,Цена: Молоко,КОСТАНАЙСКАЯ ОБЛАСТЬ,99.800000,99.201200,95.530756,90.181033,88.106870,82.732350,78.926662,79.084516,78.530924,81.358037,85.263223,86.371645,86.803503,87.237521,89.156746,89.780843,89.601282,89.242877,88.707419,90.392860,94.098968,97.957025,100.503908,102.714994,105.693729,110.767028,113.757737,115.919134,116.846488,115.911716,114.868510,110.503507,109.840486,110.829050,112.491486,113.953875,117.372491,117.020374,119.711843,118.754148,117.922869,114.621029,105.680588,104.201060,103.575854,104.818764,106.181408,110.216301,111.869546,113.771328,115.477898,115.708854,115.477436,115.361959,113.977615,114.433526,116.264462,121.612627,129.517448,129.646966,134.703197,142.111873,141.259202,138.999055,141.918035,143.053379,146.057500,145.327213,143.728613,145.884542,145.155120,149.800084,149.500483,150.098485,148.747599,149.342589,152.777469,150.638584,150.638584,151.241139,154.265961,150.409312,151.762996,158.592331,159.385293,164.963778,166.778379,175.284077,178.263906,183.077032,182.710878,186.365095,185.806000,188.593090,189.913241,196.940031,205.605393,206.016604,205.810587,216.512737,219.110890,215.386005,216.678321,215.161573,214.300927,212.800820,211.949617,211.525718,215.756232,217.913794,218.349622,219.223020,219.880689,219.880689,220.100570,219.660369,219.660369,219.440709,219.660149,221.197770
75,Молоко,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,5111.500000,4922.200000,5680.700000,6401.100000,7594.600000,7714.000000,7714.000000,7698.200000,7848.000000,7424.900000,7388.800000,8151.600000,5149.900000,5149.900000,5700.800000,6474.500000,6969.900000,7770.600000,7828.200000,7788.300000,7825.500000,7285.900000,7737.900000,8132.300000,5209.300000,5149.700000,5855.400000,6532.300000,7059.600000,7891.400000,7902.500000,7904.400000,7719.500000,7435.100000,7827.300000,8203.900000,5275.500000,5288.100000,6060.200000,6722.300000,7233.700000,7979.300000,8002.400000,8080.200000,7832.200000,7534.800000,8011.100000,8366.900000,5367.300000,5347.900000,6177.000000,6887.100000,7355.500000,8126.100000,8099.200000,8115.800000,8082.000000,7697.100000,8087.800000,8504.700000,5418.600000,5484.800000,6331.500000,7088.400000,7499.000000,8344.300000,8297.900000,8479.300000,8258.600000,7888.400000,8204.300000,8640.700000,5511.200000,5595.000000,6500.300000,7142.300000,7655.100000,8545.600000,8536.900000,8739.200000,8508.300000,8076.600000,8423.300000,8933.300000,5636.500000,5667.800000,6696.300000,7291.100000,7726.200000,8621.400000,8532.700000,8712.300000,8558.400000,8288.000000,8660.800000,9195.600000,2835.300000,2855.000000,3279.600000,3527.500000,3783.800000,4145.300000,4132.400000,4199.900000,4235.700000,4066.100000,4298.000000,14481.400000,2911.000000,2981.000000,3217.200000,3592.600000,3849.600000,4301.200000,4309.900000,4363.400000,4359.700000,4199.900000,4408.500000,5646.600000
196,Поголовье: КРС,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,251243.000000,260383.000000,266492.000000,274344.000000,281277.000000,280500.000000,269617.000000,266748.000000,263785.000000,259911.000000,256599.000000,256124.000000,2

In [5]:
# Step 1: Pivot to wide format (each indicator becomes columns of periods)
df_wide = df_milk.pivot(index="Регион", columns="Показатель")

# Step 2: Flatten multi-level columns: ('2015-01', 'КРС') → 'КРС_2015-01'
df_wide.columns = [f"{col[1]}_{col[0]}" for col in df_wide.columns]
df_wide = df_wide.reset_index()

# Step 3: Melt: one row per region-period-indicator
df_melted = df_wide.melt(id_vars="Регион", var_name="indicator_period", value_name="value")

# Step 4: Extract 'Период' and 'Показатель' from the combined column
df_melted["Период"] = df_melted["indicator_period"].str.extract(r"_(\d{4}-\d{2})$")
df_melted["Показатель"] = df_melted["indicator_period"].str.extract(r"^(.+)_\d{4}-\d{2}")

# Step 5: Pivot again to get final modeling format: one row per region+period, one column per indicator
df_milk = df_melted.pivot_table(index=["Регион", "Период"], columns="Показатель", values="value").reset_index()
print(df_milk.groupby("Регион").size().reset_index(name="Количество строк"))
df_milk

                            Регион  Количество строк
0              АКМОЛИНСКАЯ ОБЛАСТЬ               120
1              АКТЮБИНСКАЯ ОБЛАСТЬ               120
2              АЛМАТИНСКАЯ ОБЛАСТЬ               120
3               АТЫРАУСКАЯ ОБЛАСТЬ               120
4   ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               120
5                          ГАЛМАТЫ               120
6                          ГАСТАНА               120
7                         ГШЫМКЕНТ                79
8               ЖАМБЫЛСКАЯ ОБЛАСТЬ               120
9    ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               120
10          КАРАГАНДИНСКАЯ ОБЛАСТЬ               120
11            КОСТАНАЙСКАЯ ОБЛАСТЬ               120
12          КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ               120
13           МАНГИСТАУСКАЯ ОБЛАСТЬ               120
14                    ОБЛАСТЬ АБАЙ                31
15                  ОБЛАСТЬ ЖЕТІСУ                31
16                  ОБЛАСТЬ ҰЛЫТАУ                31
17            ПАВЛОДАРСКАЯ ОБЛАСТЬ            

Показатель,Регион,Период,Молоко,Осадки,Поголовье: КРС,Температура,Цена: Молоко
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,12800.2,9.8,372560.0,-12.490323,100.700000
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,15349.3,9.8,399442.0,-10.192857,100.700000
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,25086.1,8.3,425605.0,-5.870968,100.297200
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,31661.4,8.8,440023.0,4.490000,100.196903
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,49129.8,42.8,444647.0,14.574194,96.289224
...,...,...,...,...,...,...,...
2166,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-08,38641.4,0.0,1120067.0,27.874194,222.272429
2167,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-09,35950.3,0.5,1101103.0,20.766667,226.717878
2168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-10,32936.5,13.6,1078583.0,13.200000,229.211775
2169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-11,30760.6,12.1,1056289.0,6.500000,236.775763


In [6]:
df_milk = df_milk[df_milk["Регион"] != 'РЕСПУБЛИКА КАЗАХСТАН']

In [7]:
df_milk

Показатель,Регион,Период,Молоко,Осадки,Поголовье: КРС,Температура,Цена: Молоко
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,12800.2,9.8,372560.0,-12.490323,100.700000
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,15349.3,9.8,399442.0,-10.192857,100.700000
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,25086.1,8.3,425605.0,-5.870968,100.297200
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,31661.4,8.8,440023.0,4.490000,100.196903
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,49129.8,42.8,444647.0,14.574194,96.289224
...,...,...,...,...,...,...,...
2166,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-08,38641.4,0.0,1120067.0,27.874194,222.272429
2167,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-09,35950.3,0.5,1101103.0,20.766667,226.717878
2168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-10,32936.5,13.6,1078583.0,13.200000,229.211775
2169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-11,30760.6,12.1,1056289.0,6.500000,236.775763


In [8]:
df_milk.to_excel("Датасет по молоку с регрессорами.xlsx", index=False)

In [9]:
df_milk = df_milk.drop(columns=['Осадки', 'Поголовье: КРС', 'Температура', 'Цена: Молоко'])
df_milk.sample(10)

Показатель,Регион,Период,Молоко
2065,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2022-10,46495.2
614,ГАЛМАТЫ,2016-03,243.4
595,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2024-08,25528.8
1666,ОБЛАСТЬ АБАЙ,2024-09,30870.5
859,ГШЫМКЕНТ,2020-01,2362.6
2081,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2024-02,18055.5
208,АКТЮБИНСКАЯ ОБЛАСТЬ,2022-05,32088.5
1754,ПАВЛОДАРСКАЯ ОБЛАСТЬ,2016-11,20601.0
931,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2016-01,15367.1
2035,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2020-04,56941.8


In [10]:
df_milk.to_excel("Датасет по молоку.xlsx", index=False)